# Tutorial 101: Group Sequential Testing in EarlySign

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# !pip install earlysign "ibis-framework[duckdb]"

In [ ]:
import numpy as np
import pandas as pd

# Parameters
np.random.seed(42)

trials = [100] * 10
p_control = 0.01
p_treatment = 0.03

data = pd.DataFrame(
    {
        "action_date": list(range(len(trials))),
        "A_count": trials,
        "B_count": trials,
        "A_sum": [np.random.binomial(n, p_control) for n in trials],
        "B_sum": [np.random.binomial(n, p_treatment) for n in trials],
    }
)

data

In [ ]:
import ibis

from earlysign.v0.templates.ab_tests import BinomialABTest

conn = ibis.connect("duckdb://:memory:")

test = BinomialABTest(conn, "my_fantastic_experiment")

## We need to select alpha, power, max sample size, etc.
# designer = test.designer()
# designer.select()
# design.to_json()

# Use the designer to generate a valid design object
# designer = test.designer()

# Select design parameters (alpha, power, effect_size, etc.)
design = {
    "alpha": 0.05,
    "hypothesis": {"structure": "two_sided_symmetric"},
    "statistic": {"kind": "wald_z", "scale": "z"},
    "efficacy": {"style": "alpha_spending", "family": "obrien_fleming"},
    "futility": {"mode": "symmetric", "binding_mode": "non_binding"},
    "planned_max_n": 1000,
    "planned_info_times": [0.33, 0.67, 1.0],
    "effect_size": 0.01,
    "power": 0.8,
}
# Optionally, inspect or serialize the design
# print(design)
# design.to_json()
test.set_design(design)

test.plot_design()

In [ ]:
from earlysign.core.ledger import Ledger

Ledger(conn, "my_fantastic_experiment").t.select(
    "type", "payload", "attributes"
).execute()

### Design Phase
Let's first make some choices on the design of the experiment.
Toward the end, we will select the tentative sample size based on some belief on the effect size.

In [ ]:
## Let's conduct the experiment! (Here, we are only simulating it in retrospect)
## Experiment loop
for row in data.sort_values("action_date", ascending=True).to_dict(orient="records"):
    print(f"\n=== Analysis on {row['action_date']} ===")
    print(
        f"Control: {row['A_sum']}/{row['A_count']} successes ({row['A_sum'] / row['A_count']:.1%})"
    )
    print(
        f"Treatment: {row['B_sum']}/{row['B_count']} successes ({row['B_sum'] / row['B_count']:.1%})"
    )

    ## Re-instantiate test object
    test = BinomialABTest(conn, "my_fantastic_experiment")

    ## Observe data and run updates
    test.update(
        dict(nA=row["A_count"], nB=row["B_count"], mA=row["A_sum"], mB=row["B_sum"])
    )

    ## Check updated test status
    status = test.status()
    print(f"Status: {status}")
    if status.stop_recommended:
        print("⚠️ Early stopping recommended!")
        break  # This test template only has recommendations for stopping. We optionally follow it to enjoy early stopping.

    ## The process may well be reset every day
    del test

## Re-instantiate test object
test = BinomialABTest(conn, "my_fantastic_experiment")
## Check out the reports
# test.report_results()  # Check final result
# test.report_history()  # Visualize test history

In [ ]:
## Comparison: Single-shot test

## Creating Custom Templates

The built-in templates like `BinomialABTest` cover common scenarios, but you may need **custom combinations** of parameters, effect size definitions, statistics, or stopping rules for your specific domain.

### When to Create a Custom Template

You should consider creating a custom template when:

- **Custom effect size definitions**: Your field requires specialized parameterizations (e.g., clinical meaningful difference, business impact metrics, survival hazards)
- **Non-standard statistics**: You need test statistics not included in the standard library (e.g., rank-based tests, variance-weighted combinations, Bayesian posteriors)
- **Specialized stopping rules**: Your experimental context demands unique stopping logic (e.g., regulatory constraints, multi-arm rules, domain-specific futility boundaries)
- **Team standardization**: You want to encode your organization's experimental protocols into reusable, shareable files

### Benefits of Custom Templates

1. **Portability**: Templates are self-contained and can be shared across teams and projects
2. **Backend agnostic**: Same template works with DuckDB, Polars, or any ibis-supported backend
3. **Reproducibility**: Complete experimental protocol is captured in code
4. **Version control**: Templates can be stored in Git repositories
5. **Auditability**: Template definitions become part of the event log

### Template Structure

A custom template inherits from `ExperimentTemplate` and implements three key methods:

```python
from earlysign.v0.templates.base import ExperimentTemplate

class MyCustomTemplate(ExperimentTemplate):
    """Custom template for domain-specific sequential testing."""
    
    def setup(self, ledger, design_params):
        """
        Initialize experiment design and register to ledger.
        
        - Define custom effect size parameterization
        - Configure spending functions
        - Set up domain-specific boundaries
        """
        pass
    
    def step(self, ledger, observation_data):
        """
        Process one observation batch.
        
        - Compute custom statistics
        - Update decision criteria
        - Emit signals based on custom stopping rules
        """
        pass
    
    def analyze(self, ledger):
        """
        Generate analysis report from ledger events.
        
        - Extract relevant events
        - Compute summary statistics
        - Generate visualizations
        """
        pass
```

### Example: Clinical Trial with Custom Endpoints

```python
class ClinicalTrialWithQALY(ExperimentTemplate):
    """
    Sequential testing for quality-adjusted life years (QALY).
    
    - Effect size: Mean QALY difference (clinical meaningful difference = 0.5)
    - Statistic: Variance-stabilized z-score with time-to-event adjustment
    - Stopping rule: Group sequential with binding futility for ethical early stop
    """
    
    def setup(self, ledger, alpha=0.025, beta=0.10, cmd=0.5):
        # Register custom design with QALY-specific parameters
        design = {
            "effect_measure": "qaly_difference",
            "clinically_meaningful_difference": cmd,
            "alpha": alpha,
            "beta": beta,
            "spending_function": "obrien_fleming",
            "binding_futility": True
        }
        ledger.write_event(namespace="design", kind="registered", 
                          type="QALYDesign", data=design)
    
    def step(self, ledger, qaly_data):
        # Compute variance-stabilized statistic
        # Update group sequential boundaries
        # Check stopping rules with ethical considerations
        pass
    
    def analyze(self, ledger):
        # Generate QALY-specific reports and visualizations
        pass
```

### Using Custom Templates Across Backends

Once created, your template is portable across different backends:

```python
import ibis

# Create template instance
template = ClinicalTrialWithQALY()

# Use with DuckDB
conn_duckdb = ibis.connect("duckdb://data.db")
ledger_duck = Ledger(conn_duckdb, "clinical_trial_001")
template.setup(ledger_duck, alpha=0.025, beta=0.10, cmd=0.5)

# Use with Polars (same template!)
conn_polars = ibis.polars.connect()
ledger_polars = Ledger(conn_polars, "clinical_trial_002")
template.setup(ledger_polars, alpha=0.025, beta=0.10, cmd=0.5)
```

### Best Practices

1. **Document thoroughly**: Include docstrings explaining the statistical rationale and domain assumptions
2. **Validate inputs**: Check parameter constraints and raise informative errors
3. **Use typed payloads**: Define clear payload schemas for your custom event types
4. **Test across backends**: Verify your template works with multiple ibis backends
5. **Version your templates**: Use semantic versioning for template definitions stored in the ledger

For more details on the event-sourcing architecture and component design, see [Explanation: Concepts](../explanation/concepts.md).